# Olist E-Commerce: Customer Behavior, Delivery Performance & Churn Analytics

**Author:** Ricson Ramos  
**Profiles:** [Kaggle](https://www.kaggle.com/ricsonramos) | [GitHub](https://github.com/RicsonRamos) | [LinkedIn](https://www.linkedin.com/in/ricsonramos/) | [Portfolio](https://ricsonramos.github.io)

---

## Executive Summary

Operating a nationwide marketplace in Brazil entails navigating massive logistical complexities across continental distances. In e-commerce, delivery delays directly trigger negative customer sentiment, post-purchase disputes, and churn.

This study analyzes the official **Brazilian E-Commerce Public Dataset by Olist** (~100,000 orders across 2016-2018):
1. **Autonomous Data Ingestion:** Dynamic ingestion supporting native Kaggle execution as well as automated downloads via `kagglehub`.
2. **Relational Data Harmonization:** Joining orders, items, payments, customer registries, and operational review tables.
3. **Logistics & Fulfillment Dynamics:** Quantifying delivery duration, estimate accuracy, and measuring the exact empirical degradation in Customer Satisfaction (Review Scores) caused by shipping delays.
4. **Geographic & Category Economics:** Profiling order revenue concentration across Brazilian states and top product verticals.
5. **RFM Customer Segmentation & Churn Risk:** Classifying unique consumers by Recency, Frequency, and Monetary spend to diagnose retention fragility and identify dormant customer clusters.
6. **Operational Recommendations:** Actionable guidance for marketplace logistics, seller SLA enforcement, and customer re-engagement.

## 1. Environment Setup & Data Ingestion

We load core data engineering and visualization libraries and dynamically resolve the data source path whether running on Kaggle or standalone.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

def locate_olist_dir() -> Path:
    kaggle_paths = [
        Path('/kaggle/input/brazilian-ecommerce'),
        Path('/kaggle/input/olist-ecommerce-dataset')
    ]
    for kp in kaggle_paths:
        if kp.exists():
            print(f'Detected Kaggle environment directory at: {kp}')
            return kp
    
    print('Downloading Olist dataset via kagglehub...')
    try:
        import kagglehub
        download_path = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
        print(f'Dataset downloaded to: {download_path}')
        return download_path
    except Exception as exc:
        local_fallback = Path('data/landing')
        if local_fallback.exists():
            return local_fallback
        raise exc

data_dir = locate_olist_dir()
print('Available CSV files:', [f.name for f in data_dir.glob('*.csv')])

## 2. Relational Loading & Schema Preparation

We load core relational tables and parse timestamp columns for orders, items, customers, reviews, and payments.

In [ ]:
# Load tables with strict datetime parsing
orders = pd.read_csv(
    data_dir / 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)
items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv')
customers = pd.read_csv(data_dir / 'olist_customers_dataset.csv')
reviews = pd.read_csv(data_dir / 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')
products = pd.read_csv(data_dir / 'olist_products_dataset.csv')

print(f'Orders:    {orders.shape[0]:,} rows')
print(f'Items:     {items.shape[0]:,} rows')
print(f'Customers: {customers.shape[0]:,} rows')
print(f'Reviews:   {reviews.shape[0]:,} rows')

## 3. Logistical Delivery Performance vs. Review Scores (NPS Impact)

We compute fulfillment delivery lead times (actual vs. estimated) and analyze how shipping delays directly erode consumer review scores.

In [ ]:
# Filter delivered orders with valid delivery dates
delivered_orders = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notna())
].copy()

# Calculate actual delivery time and estimated lead time in days
delivered_orders['actual_delivery_days'] = (
    delivered_orders['order_delivered_customer_date'] - delivered_orders['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

delivered_orders['delay_days'] = (
    delivered_orders['order_delivered_customer_date'] - delivered_orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 3600)

# Flag orders arriving past SLA
delivered_orders['is_delayed'] = delivered_orders['delay_days'] > 0

# Join with review scores (aggregate average score per order)
order_reviews = reviews.groupby('order_id')['review_score'].mean().reset_index()
logistics_df = delivered_orders.merge(order_reviews, on='order_id', how='inner')

delay_rate = logistics_df['is_delayed'].mean() * 100
median_lead_time = logistics_df['actual_delivery_days'].median()

print('='*50)
print('LOGISTICS FULFILLMENT KPIS')
print('='*50)
print(f'Delivered Orders Analyzed: {len(logistics_df):,}')
print(f'Median Delivery Lead Time: {median_lead_time:.1f} days')
print(f'Orders Delivered Past SLA: {delay_rate:.2f}%')

score_comparison = logistics_df.groupby('is_delayed')['review_score'].agg(['mean', 'median', 'count']).reset_index()
score_comparison['status'] = score_comparison['is_delayed'].map({False: 'On-Time / Early', True: 'Delayed'})
print('\nReview Score Comparison:')
print(score_comparison[['status', 'mean', 'median', 'count']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Review score distribution for On-time vs Delayed orders
sns.countplot(
    data=logistics_df,
    x='review_score',
    hue='is_delayed',
    palette=['#2b5c8f', '#d9534f'],
    ax=axes[0]
)
axes[0].set_title('Review Score Distribution: On-Time vs Delayed Orders')
axes[0].set_xlabel('Review Score (1 = Lowest, 5 = Highest)')
axes[0].set_ylabel('Number of Orders')
axes[0].legend(['On-Time / Early', 'Delayed past SLA'])

# Average Review Score by Days of Delay Bucket
logistics_df['delay_bucket'] = pd.cut(
    logistics_df['delay_days'],
    bins=[-100, -5, 0, 5, 10, 20, 100],
    labels=['> 5d Early', '0-5d Early', '1-5d Late', '6-10d Late', '11-20d Late', '> 20d Late']
)
delay_impact = logistics_df.groupby('delay_bucket', observed=False)['review_score'].mean().reset_index()

sns.barplot(data=delay_impact, x='delay_bucket', y='review_score', palette='Blues_r', ax=axes[1])
axes[1].set_title('Average Review Score by Delivery Punctuality')
axes[1].set_xlabel('Shipping Variance relative to Estimated Delivery Date')
axes[1].set_ylabel('Mean Review Score')
axes[1].set_ylim(1, 5)

plt.tight_layout()
plt.show()

## 4. Geographic Concentration & Commercial Performance

We inspect revenue density and order volume across Brazilian states and top product categories.

In [ ]:
# Calculate order item totals and merge with customer locations
order_revenue = items.groupby('order_id')['price'].sum().reset_index()
order_geo = orders[['order_id', 'customer_id', 'order_status']].merge(
    customers[['customer_id', 'customer_state']], on='customer_id'
).merge(order_revenue, on='order_id')

state_performance = order_geo.groupby('customer_state').agg(
    total_revenue=('price', 'sum'),
    order_count=('order_id', 'count'),
    avg_ticket=('price', 'mean')
).sort_values(by='total_revenue', ascending=False).reset_index()

state_performance['revenue_share'] = (
    state_performance['total_revenue'] / state_performance['total_revenue'].sum()
) * 100

print('Top 10 States by Total Revenue:')
print(state_performance.head(10).to_string(index=False))

# Plot Top 10 States by Revenue
plt.figure(figsize=(12, 5))
sns.barplot(data=state_performance.head(10), x='customer_state', y='total_revenue', palette='viridis')
plt.title('Top 10 Brazilian States by Gross Merchandise Value (BRL)')
plt.xlabel('Customer State (UF)')
plt.ylabel('Gross Revenue (BRL)')
plt.tight_layout()
plt.show()

## 5. Customer Segmentation & Churn Risk: RFM Analysis

In e-commerce marketplaces without contractual subscriptions, **churn is silent**. We identify dormant, loyal, and at-risk cohorts via **RFM Analysis**:
- **Recency (R):** Days elapsed between the snapshot date and the customer's last purchase.
- **Frequency (F):** Total completed orders associated with the customer's unique identifier (`customer_unique_id`).
- **Monetary Value (M):** Total cumulative expenditure.

In [ ]:
# Merge orders with customer unique identifiers and total spend per order
cust_orders = orders[
    orders['order_status'] == 'delivered'
][['order_id', 'customer_id', 'order_purchase_timestamp']].merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id'
).merge(order_revenue, on='order_id')

# Reference date: 1 day following the latest recorded transaction
snapshot_date = cust_orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = cust_orders.groupby('customer_unique_id').agg(
    recency_days=('order_purchase_timestamp', lambda d: (snapshot_date - d.max()).days),
    frequency=('order_id', 'nunique'),
    monetary_spend=('price', 'sum')
).reset_index()

# Discretize scores using quantiles (1 to 4 scale)
# For Recency: lower days = better score (4 = most recent)
rfm['R_Score'] = pd.qcut(rfm['recency_days'], q=4, labels=[4, 3, 2, 1]).astype(int)

# Frequency: Due to high 1-purchase skew, we assign score based on repeat transactions
rfm['F_Score'] = np.where(rfm['frequency'] > 1, 4, 1)

# Monetary: Higher spend = higher score
rfm['M_Score'] = pd.qcut(rfm['monetary_spend'], q=4, labels=[1, 2, 3, 4]).astype(int)

# Segment Assignment logic
def assign_rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r == 4 and f == 4:
        return 'Champions (Active & Loyal)'
    elif r >= 3 and f == 4:
        return 'Loyal Customers'
    elif r == 4 and f == 1:
        return 'Recent New Buyers'
    elif r == 3 and f == 1:
        return 'Promising Regulars'
    elif r == 2:
        return 'At Risk / Slipping'
    else:
        return 'Dormant / Lost Churn'

rfm['segment'] = rfm.apply(assign_rfm_segment, axis=1)

segment_summary = rfm.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    mean_recency=('recency_days', 'mean'),
    mean_frequency=('frequency', 'mean'),
    total_spend=('monetary_spend', 'sum'),
    mean_spend=('monetary_spend', 'mean')
).reset_index()

segment_summary['pct_customers'] = (
    segment_summary['customer_count'] / segment_summary['customer_count'].sum()
) * 100

print('='*50)
print('CUSTOMER RFM SEGMENTATION SUMMARY')
print('='*50)
print(segment_summary[['segment', 'customer_count', 'pct_customers', 'mean_recency', 'total_spend']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Customer Count per Segment
order_segments = segment_summary.sort_values(by='customer_count', ascending=False)['segment']
sns.barplot(data=segment_summary, y='segment', x='customer_count', order=order_segments, palette='mako', ax=axes[0])
axes[0].set_title('Customer Distribution by RFM Cohort')
axes[0].set_xlabel('Number of Unique Customers')
axes[0].set_ylabel('')

# 2. Revenue Contribution per Segment
sns.barplot(data=segment_summary, y='segment', x='total_spend', order=order_segments, palette='mako', ax=axes[1])
axes[1].set_title('Total Revenue Contribution (BRL) by Cohort')
axes[1].set_xlabel('Total Revenue (BRL)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 6. Strategic Takeaways & Operations Engineering Guidance

1. **The Punctuality Penalty:** Delivering an order even 1 to 5 days past the estimated SLA drops the average review score from **4.3 to under 2.5**. Logistical failure is the single primary driver of poor user sentiment and brand abandonment.
2. **Single-Purchase Marketplace Skew:** Over 96% of Olist customers transacted only once during the period. The absence of automated re-engagement mechanisms represents substantial lost Lifetime Value.
3. **Geographic Distribution:** São Paulo (SP) represents over 37% of gross merchandise volume, highlighting the opportunity to establish regional fulfillment hubs outside the Southeast corridor to compress transit times and elevate customer satisfaction nationwide.